# Capture activation traces

Run from this notebook's directory. One dense forward per prompt records projection-input importance and saves `activation_traces.npz`. Continue with [02_global_oracle.ipynb](02_global_oracle.ipynb) to generate the comparison CSV, then [02_analyze_oracle.ipynb](02_analyze_oracle.ipynb) for the figures.

PyTorch and Transformers load the checkpoint below; an uncached checkpoint is downloaded.

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TRACE = Path("activation_traces.npz")
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
PROMPTS = (
    "Explain why contiguous flash reads can be faster than scattered reads.",
    "Summarize the trade-off between neural-network sparsity and accuracy.",
    "Write a short Python function that computes a moving average.",
)

## 1. Load the model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
net = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.bfloat16).to(device).eval()

## 2. Record projection inputs

A pre-forward hook records $v_i=\operatorname{mean}_{\rm batch,token}|x_i|$ at each attention or MLP projection. Absolute values and means are computed in the activation dtype, then converted to float32. The dense forward uses the original activations.

In [ ]:
def record(name):
    def hook(layer, inputs):
        n, d = layer.in_features, layer.out_features
        key = f"values_{len(entries):06d}"
        arrays[key] = inputs[0].detach().abs().reshape(-1, n).mean(0).float().cpu().numpy()
        entries.append(dict(trace_id=len(entries), input_index=input_index,
            module=name, n=n, d=d, shape=f"{n}x{d}", array=key,
            num_tokens=int(tokens["input_ids"].numel())))
    return hook

In [ ]:
entries, arrays, hooks = [], {}, []
try:
    for name, layer in net.named_modules():
        if isinstance(layer, torch.nn.Linear) and ".layers." in name and name.endswith("_proj"):
            hooks.append(layer.register_forward_pre_hook(record(name)))
    with torch.inference_mode():
        for input_index, prompt in enumerate(PROMPTS):
            text = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
            tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
            net(**tokens, use_cache=False)
            print(f"{input_index + 1}/{len(PROMPTS)} prompts captured", end="\r")
finally:
    for handle in hooks:
        handle.remove()

## 3. Save the archive

Store the vectors with projection dimensions, prompt indices, model revision and dtype.

In [ ]:
manifest = dict(model=MODEL_ID, model_revision=MODEL_REVISION,
    model_dtype=str(next(net.parameters()).dtype).split(".")[-1],
    prompts=list(PROMPTS), prompt_count=len(PROMPTS), traces=entries, trace_count=len(entries))
np.savez_compressed(TRACE, manifest_json=np.asarray(json.dumps(manifest)), **arrays)
manifest["trace_count"]